# Reddit Comment-to-Post Attachment
<!-- structured-notebook -->
## Notebook Summary
Purpose: link Reddit comments to their parent posts so sentiment analysis can operate on the post-comment pair rather than just the post text.

Main steps:
- Load comments and parent posts from the decompressed dumps.
- Join comments to posts on `link_id` / `parent_id`.
- Compute and save per-post aggregate comment sentiment alongside post-level sentiment.

Most comments come from `r/Biohackers`, which has the highest engagement volume of the three retained subreddits. A caveat from the project journal: Reddit's `score` field is upvotes minus downvotes, not raw upvotes — Reddit itself does not expose the raw upvote count.


# Inputs / Outputs
<!-- io-table -->

| Role | File | Upstream / downstream |
|---|---|---|
| Input | `<DATA_ROOT>/Reddit/input/raw/` | Raw comment dumps |
| Input | `<DATA_ROOT>/Reddit/output/merged_submissions.jsonl` | Parent-post metadata |
| Output | `<DATA_ROOT>/Reddit/output/comments/discussion_comments_linked.csv` | Post-comment pairs with metadata |
| Output | `<DATA_ROOT>/Reddit/output/comments/discussion_post_comment_sentiment.csv` | Consumed by `RQ2.ipynb` |


In [ ]:
from src.shared_reddit_telegram.config import (
    REDDIT_OUTPUT,
)
import pandas as pd
import numpy as np
import re

# --- Update these paths ---
SUBMISSIONS_PATH = REDDIT_OUTPUT / "merged_submissions.csv"   # or .jsonl
COMMENTS_PATH   = REDDIT_OUTPUT / "merged_comments.jsonl"       # your file

In [ ]:
def read_any(path):
    if path.endswith(".jsonl"):
        return pd.read_json(path, lines=True)
    return pd.read_csv(path)

subs = read_any(SUBMISSIONS_PATH)
coms = read_any(COMMENTS_PATH)

# Timestamps
for df in (subs, coms):
    if "created_utc" in df.columns:
        df["created_utc"] = pd.to_datetime(df["created_utc"], unit="s", errors="coerce")

# Helpers
def strip_prefix(s):
    # remove t1_/t3_/etc prefixes from IDs
    return s.astype(str).str.replace(r"^t[0-6]_", "", regex=True)

# Post IDs in submissions
if "id" in subs.columns:
    subs["post_id"] = subs["id"].astype(str)

# Post IDs in comments (via link_id or permalink)
if "link_id" in coms.columns:
    coms["root_post_id"] = strip_prefix(coms["link_id"])
elif "permalink" in coms.columns:
    # /r/<sub>/comments/<POST_ID>/<slug>/<COMMENT_ID>/
    coms["root_post_id"] = coms["permalink"].str.extract(r"/comments/([a-z0-9]+)/", expand=False)
else:
    raise ValueError("No way to recover post id from comments: need link_id or permalink.")

    Filtering for 'Discussion' posts

In [ ]:
# common flair columns: link_flair_text or flair_text (depends on your dump)
flair_col = "link_flair_text" if "link_flair_text" in subs.columns else (
    "flair_text" if "flair_text" in subs.columns else None
)

if flair_col is None:
    raise ValueError("Your submissions file must include a flair column (e.g., link_flair_text).")

# Case-insensitive match for 'discussion' (also catches 'weekly discussion', etc.)
subs["is_discussion"] = subs[flair_col].astype(str).str.contains(r"\bdiscussion\b", case=False, na=False)

discussion_posts = subs.loc[subs["is_discussion"]].copy()

print(f"Found {len(discussion_posts):,} 'Discussion' posts out of {len(subs):,} total.")

Pull comments for discussion posts

In [ ]:
# Join via post id
disc_post_ids = set(discussion_posts["post_id"].astype(str))
disc_comments = coms[coms["root_post_id"].isin(disc_post_ids)].copy()

print(f"Matched {len(disc_comments):,} comments on 'Discussion' posts.")

In [ ]:
print(f"Number of comments overall: {len(coms)}")

In [ ]:
def coerce_datetime_reddit(series: pd.Series) -> pd.Series:
    """
    Parse Reddit 'created' fields robustly:
      - Accepts ISO strings
      - Detects Unix epoch in s / ms / us / ns via magnitude
      - Falls back across units if we accidentally get ~1970
    Returns tz-aware UTC datetimes.
    """
    # First try normal parsing (works for ISO strings)
    dt = pd.to_datetime(series, errors="coerce", utc=True)

    # If we already have plenty of good datetimes not around 1970, keep them
    if dt.notna().mean() > 0.5:
        yrs = dt.dropna().dt.year
        if not yrs.empty and (yrs.le(1971).mean() < 0.7):
            return dt

    # Numeric epoch detection
    num = pd.to_numeric(series, errors="coerce")
    if num.notna().sum() == 0:
        return dt  # nothing better

    med = float(np.nanmedian(np.abs(num.values)))

    # Magnitude thresholds (use tight ranges)
    # 1e9–1e10: seconds, 1e12–1e13: ms, 1e15–1e16: us, 1e18–1e19: ns
    if 1e18 <= med < 1e20:
        unit = "ns"
    elif 1e15 <= med < 1e17:
        unit = "us"
    elif 1e12 <= med < 1e14:
        unit = "ms"
    elif 1e9 <= med < 1e11:
        unit = "s"
    else:
        # If in-between, try ms first (most Reddit dumps use seconds or ms)
        unit = "ms"

    dt2 = pd.to_datetime(num, unit=unit, errors="coerce", utc=True)

    # If we got mostly ~1970, auto-correct by trying the next smaller unit
    def looks_like_1970(dts: pd.Series) -> bool:
        yrs = dts.dropna().dt.year
        return (not yrs.empty) and (yrs.le(1971).mean() > 0.7)

    if looks_like_1970(dt2):
        fallback = {"ns": "us", "us": "ms", "ms": "s", "s": "ms"}  # try ‘smaller’ (or ms from s)
        alt = fallback.get(unit, "ms")
        dt3 = pd.to_datetime(num, unit=alt, errors="coerce", utc=True)
        if dt3.notna().sum() >= dt2.notna().sum():
            dt2 = dt3

    # Prefer the parse that yields more real timestamps
    if dt2.notna().sum() > dt.notna().sum():
        return dt2
    return dt

In [ ]:
import matplotlib.pyplot as plt
import os

In [ ]:
dt = df["created_utc"]

df["_date"] = dt
df["_year"] = dt.dt.year
df["_dow"]  = dt.dt.day_name()
df["_hour"] = dt.dt.hour

out_dir = REDDIT_OUTPUT / "comments" / "figs"

# Yearly (bars; robust to NA)
tmp = df.loc[df["_year"].notna()].copy()
tmp["_year"] = pd.to_numeric(tmp["_year"], errors="coerce").astype("Int64")
yearly = (tmp.groupby("_year", dropna=True).size().rename("post_count").sort_index())
yearly.to_csv(os.path.join(out_dir, "yearly_counts.csv"))

ax = yearly.plot(kind="bar", title="Reddit: Comments per year")
ax.set_xlabel("Year")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, "yearly_counts.png"), dpi=200, bbox_inches="tight", facecolor="white")
plt.close()

Quick EDA

In [ ]:
def monthly_counts(df, timestamp_col="created_utc"):
    tmp = df.dropna(subset=[timestamp_col]).copy()
    tmp["month"] = tmp[timestamp_col].dt.to_period("M").dt.to_timestamp()
    return tmp.groupby("month").size().rename("count").reset_index()

disc_posts_m = monthly_counts(discussion_posts, "created_utc")
disc_coms_m  = monthly_counts(disc_comments, "created_utc")

# Inspect tables (plot later in your notebook/BI tool)
print(disc_posts_m.head())
print(disc_coms_m.head())

In [ ]:
top_subs_posts = (discussion_posts["subreddit"]
                  .value_counts()
                  .head(20)
                  .rename_axis("subreddit")
                  .reset_index(name="discussion_posts"))

top_subs_comments = (disc_comments["subreddit"]
                     .value_counts()
                     .head(20)
                     .rename_axis("subreddit")
                     .reset_index(name="comments_on_discussions"))

print(top_subs_posts.head(10))
print(top_subs_comments.head(10))

# If submissions contain num_comments/score:
if "num_comments" in discussion_posts.columns:
    top_threads = (discussion_posts
                   .nlargest(20, "num_comments")[["post_id", "subreddit", "title", "num_comments"]]
                   .fillna(""))
else:
    # fallback via joined comments
    top_threads = (disc_comments.groupby(["root_post_id", "subreddit"])
                   .size()
                   .reset_index(name="comment_count")
                   .nlargest(20, "comment_count"))
print(top_threads.head(10))

In [ ]:
# Controversial vs non-controversial
contr = (disc_comments["controversiality"] == 1).mean()
print(f"Share of controversial comments: {contr:.1%}")

# OP presence in discussion threads
op_share = (disc_comments["is_submitter"] == True).mean()
print(f"Share of comments written by OPs (is_submitter): {op_share:.1%}")

# Comment length distribution
disc_comments["char_len"] = disc_comments["body"].fillna("").str.len()
len_stats = disc_comments["char_len"].describe(percentiles=[.5,.9,.99]).round(1)
print(len_stats)

In [ ]:
# pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
an = SentimentIntensityAnalyzer()
disc_comments["vader"] = disc_comments["body"].fillna("").map(lambda t: an.polarity_scores(t)["compound"])

print(disc_comments["vader"].describe())

In [ ]:
# Save comments linked to discussion posts
comments_out_path = REDDIT_OUTPUT / "comments" / "discussion_comments_linked.csv"
disc_comments.to_csv(comments_out_path, index=False)
print(f"Saved linked discussion comments to {comments_out_path}")

# Sentiment analysis for posts and their comments
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
an = SentimentIntensityAnalyzer()

def get_sentiment(text):
    if pd.isna(text):
        return np.nan
    return an.polarity_scores(str(text))["compound"]

# Sentiment for posts
if "title" in discussion_posts.columns:
    discussion_posts["post_sentiment"] = discussion_posts["title"].fillna("").map(get_sentiment)
else:
    discussion_posts["post_sentiment"] = np.nan

# Sentiment for comments (already computed above as 'vader')
# Aggregate sentiment for each post
agg_funcs = {
    "vader": ["mean", "std", "count", "median", "min", "max"]
}
comment_sentiment_summary = disc_comments.groupby("root_post_id").agg(agg_funcs)
comment_sentiment_summary.columns = ["_".join(col) for col in comment_sentiment_summary.columns]
comment_sentiment_summary = comment_sentiment_summary.reset_index()

# Merge post sentiment and comment sentiment summary
sentiment_output = discussion_posts[["post_id", "subreddit", "title", "score", "post_sentiment"]].merge(
    comment_sentiment_summary, left_on="post_id", right_on="root_post_id", how="left"
)

# Combine and normalize score and comment count for better ranking
sentiment_output["norm_score"] = (sentiment_output["score"] - sentiment_output["score"].min()) / (sentiment_output["score"].max() - sentiment_output["score"].min())
sentiment_output["norm_comment_count"] = (sentiment_output["vader_count"] - sentiment_output["vader_count"].min()) / (sentiment_output["vader_count"].max() - sentiment_output["vader_count"].min())
sentiment_output["combined_rank"] = 0.5 * sentiment_output["norm_score"] + 0.5 * sentiment_output["norm_comment_count"]

# Save output with new ranking metric
sentiment_out_path = REDDIT_OUTPUT / "comments" / "discussion_post_comment_sentiment.csv"
sentiment_output.to_csv(sentiment_out_path, index=False)
print(f"Saved sentiment analysis summary with combined ranking to {sentiment_out_path}")


In [ ]:
import matplotlib.pyplot as plt
# Bar plot: mean sentiment for each post (with std as error bars)
# Only for posts with at least 5 comments for robustness
sentiment_plot_df = sentiment_output[sentiment_output["vader_count"] >= 5].copy()

plt.figure(figsize=(12, 6))
plt.bar(sentiment_plot_df["title"], sentiment_plot_df["vader_mean"], yerr=sentiment_plot_df["vader_std"], alpha=0.7)
plt.xticks(rotation=90)
plt.xlabel("Discussion Post Title")
plt.ylabel("Mean Sentiment (Comments)")
plt.title("Mean Sentiment of Comments per Discussion Post (Error = Std)")
plt.tight_layout()
plt.savefig(REDDIT_OUTPUT / "comments" / "discussion_post_comment_sentiment_barplot.png", dpi=200)
plt.close()
print("Saved sentiment bar plot with std error bars.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

sentiment_output = pd.read_csv(REDDIT_OUTPUT / "comments" / "discussion_post_comment_sentiment.csv")
print(sentiment_output)

# Improved, visually clear bar plot: Top 20 posts by combined rank
N = 20
sentiment_plot_df = sentiment_output[sentiment_output["vader_count"] >= 5].copy()
sentiment_plot_df = sentiment_plot_df.nlargest(N, "combined_rank")
sentiment_plot_df = sentiment_plot_df.sort_values("combined_rank", ascending=False)

# Truncate long titles for readability
sentiment_plot_df["short_title"] = sentiment_plot_df["title"].str.slice(0, 60) + "..."

# Reverse the order of the short_title axis for correct ranking display
sentiment_plot_df = sentiment_plot_df.iloc[::-1]

plt.figure(figsize=(13, 10))
colors = plt.cm.RdYlGn((sentiment_plot_df["vader_mean"] + 1) / 2)
ax1 = plt.gca()
bars = ax1.barh(sentiment_plot_df["short_title"], sentiment_plot_df["vader_mean"], xerr=sentiment_plot_df["vader_std"], color=colors, alpha=0.85, edgecolor='black', label="Mean Sentiment")
ax1.set_xlabel("Mean Sentiment (Comments)", fontsize=14)
ax1.set_ylabel("Discussion Post Title", fontsize=14)
ax1.set_title(f"Top {N} Discussion Posts by Combined Rank\nMean Sentiment (Error = Std) and Score", fontsize=16, fontweight='bold')
ax1.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()

# Adjust padding for annotations
xmax = sentiment_plot_df["vader_mean"].max() + sentiment_plot_df["vader_std"].max() + 0.1  # Reduced padding
for i, (mean, std) in enumerate(zip(sentiment_plot_df["vader_mean"], sentiment_plot_df["vader_std"])):
    score_val = sentiment_plot_df["score"].iloc[i] if "score" in sentiment_plot_df.columns else None
    comment_count = sentiment_plot_df["vader_count"].iloc[i]
    combined_rank = sentiment_plot_df["combined_rank"].iloc[i]
    label = f"Sent: {mean:.2f} | Score: {score_val} | Comments: {comment_count} | Rank: {combined_rank:.2f}"
    plt.text(xmax, i, label, va='center', ha='left', fontsize=10, color='dimgray', fontweight='bold')

# Add extra padding to ensure text is not cut off
plt.savefig(REDDIT_OUTPUT / "comments" / "discussion_post_comment_sentiment_barplot_top20_pretty.png", dpi=200, bbox_inches="tight", pad_inches=1)
plt.show()
plt.close()
print(f"Saved visually improved sentiment bar plot for top {N} posts by combined rank.")


In [ ]:
# Save filtered discussion posts for future use
discussion_posts_out_path = REDDIT_OUTPUT / "comments" / "discussion_posts_filtered.csv"
discussion_posts.to_csv(discussion_posts_out_path, index=False)
print(f"Saved filtered discussion posts to {discussion_posts_out_path}")


---
<!-- nav-footer -->

← [06 pkl csv conversion](../../../../../SocialMedia/Reddit/notebooks/BERTopic/04_topic_matching/06_pkl_csv_conversion.ipynb) | [Project Overview](../../../../../PROJECT_OVERVIEW.ipynb)
